In [ ]:
# ====================================================
# Libraries
# ====================================================

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp, trapezoid, cumulative_trapezoid
from scipy.interpolate import RegularGridInterpolator
from scipy.special import erf
from multiprocessing import Pool
import os
import time
import emcee
from getdist import MCSamples, plots
import matplotlib.pyplot as plt

In [2]:
# =========================
# Loading FRB data
# =========================

frb_dat_path = "/home/brunowesley/projetos/MCMC-cosmo/Data/FRB/frb_catalog.txt"
df_frb = pd.read_csv(frb_dat_path, sep=r"\t", engine='python')
df_frb = df_frb.sort_values(by="Redshift", ascending=True)
df_frb = df_frb.reset_index(drop=True)

z_frb   = df_frb["Redshift"].to_numpy(float)
DM_obs  = df_frb["DM_obs"].to_numpy(float)
DM_MW   = df_frb["DM_MW_ISM_(NE2001)"].to_numpy(float)
DM_halo_mean = 50
DM_MW_total = DM_MW + DM_halo_mean
DM_ext_obs = DM_obs - DM_MW_total

In [3]:
# ====================================================
# Fiducial / Priors
# ====================================================
 
# Hard bounds
H0_min,       H0_max       = 50.0,  90.0
Om0_min,      Om0_max      = 0.1,   0.6
Ob_min,       Ob_max       = 0.01,  0.1
lbd_min,      lbd_max      = 1.54,  10.0
sig_host_min, sig_host_max = 0.2,   2.0
e_mu_min,     e_mu_max     = 20.0,  200.0
 
# Physical constants
c_kms = 299792.458         # speed of light [km/s]
c_si = 2.99792458e8        # speed of light [m/s] (SI)
G_si = 6.67430e-11
mp_si = 1.67262192369e-27

# Cosmological/astrophysical parameters
f_IGM = 0.83
chi_e = 0.875

# HS model index (fixed)
n = 1

# Scalaron mass scale
delta_s = 1e-7

# Unit conversion factors
km_to_pc = 1.0 / (3.0857e13)
m2_to_cm2 = 1e4
Mpc_to_cm = 3.0857e24


# ====================================================
# Redshift / scale-factor / number of e-folds grids
# ====================================================

ai,    af    = 0.2,        1.0
zi,    zf    = 0.0,        4.0
ln_ai, ln_af = np.log(ai), np.log(af)

N_grid = 300
z_grid = np.linspace(zi, zf, N_grid)

In [4]:
# ====================================================
# Custom exception for ODE failure
# ====================================================

class ODEFailure(Exception):
    pass


# ====================================================
# ODE system for yH(t=lna) in ST f(R) gravity
# ====================================================

def yH_ST_ode(lna, y, H0, Om0, lbd):
    
    yH, YH = y
    a = np.exp(lna)
    
    ms  = H0**2 * Om0
    Rs  = 6.0 * H0**2 * (1.0 - Om0) / lbd
    M2  = Rs / delta_s
    
    R = 3.0 * ms * (YH + 4*yH + a**(-3))
    
    ratio = R / Rs
    ratio2 = ratio**2
    term_base = 1.0 + ratio2
    
    f_R = R + lbd * Rs * (term_base**(-n) - 1) + R**2 / (6 * M2)
    fR = 1.0 - 2.0 * n * lbd * ratio * term_base**(-n-1) + R / (3 * M2)
    fRR = -(2.0 * n * lbd / Rs) * term_base**(-n-1) * ( 1.0 - 2*(n + 1) * ratio2 * term_base**(-1) ) + 1.0 / (3 * M2)
    
    if fRR == 0.0 or not np.isfinite(fRR):
        raise ODEFailure(f"fRR singular at lna={lna:.4f}")
    
    yaux1 = yH + a**(-3)
    yaux2 = a**(-3)
    yaux3 = ((1 - fR) * yaux2) + ((R - f_R) / (3 * ms))
    
    J1 = 4.0 + (1.0 / yaux1) * ((1.0 - fR) / (6.0 * ms * fRR))
    J2 = (1.0 / yaux1) * ((2.0 - fR) / (3.0 * ms * fRR))
    J3 = -(3.0 * yaux2) - (yaux3 / yaux1) * (1.0 / (6.0 * ms * fRR))
    
    dYH = -J1 * YH - J2 * yH - J3
    return [YH, dYH]


# ====================================================
# ODE integration → H(lna)
# ====================================================

def H_ST_lna(lna_eval, H0, Om0, lbd):
    ms  = H0**2 * Om0
    yH0 = (1.0 - Om0) / Om0
    YH0 = 0.0
    y0  = [yH0, YH0]
    try:
        sol = solve_ivp(
            yH_ST_ode,
            (ln_ai, ln_af),
            y0,
            t_eval=lna_eval,
            method="LSODA",
            rtol=1e-6,
            atol=1e-8,
            args=(H0, Om0, lbd),
            dense_output=False
        )
    except Exception:
        return None
    if sol.status < 0 or np.any(~np.isfinite(sol.y)):
        return None
    a_eval = np.exp(lna_eval)
    yH_sol = sol.y[0]
    H_sol  = np.sqrt(ms * (yH_sol + a_eval**(-3)))
    return H_sol


# ====================================================
# H(z) via lna = - ln(1+z) conversion
# ====================================================

def H_ST_z(z_eval, H0, Om0, lbd):
    z_eval    = np.atleast_1d(np.asarray(z_eval, dtype=float))
    lna_eval  = - np.log(1.0 + z_eval)
    in_domain = (lna_eval >= ln_ai) & (lna_eval <= ln_af)
    if not np.any(in_domain):
        return None
    valid_idx   = np.where(in_domain)[0]
    lna_valid   = lna_eval[valid_idx]
    sort_idx    = np.argsort(lna_valid)
    lna_sorted  = lna_valid[sort_idx]
    unique_mask = np.concatenate(([True], np.diff(lna_sorted) > 1e-12))
    lna_unique  = lna_sorted[unique_mask]
    H_unique    = H_ST_lna(lna_unique, H0, Om0, lbd)
    if H_unique is None:
        return None
    if not np.all(unique_mask):
        H_at_sorted = np.interp(lna_sorted, lna_unique, H_unique)
    else:
        H_at_sorted = H_unique
    H_valid           = np.empty(len(lna_valid))
    H_valid[sort_idx] = H_at_sorted
    H_z               = np.full(len(z_eval), np.nan)
    H_z[valid_idx]    = H_valid
    return H_z

In [5]:
# ====================================================
# Theoretical IGM dispersion measure [pc/cm^3]
# ====================================================

def DM_IGM_ST(z, H0, Om0, Ob, lbd):
    prefactor   = 3.0 * c_si * f_IGM * chi_e / (8.0 * np.pi * G_si * mp_si)
    H_grid      = H_ST_z(z_grid, H0, Om0, lbd)
    integrand   = (1 + z_grid) / H_grid
    I_grid      = cumulative_trapezoid(integrand, z_grid, initial=0.0)
    I_z         = np.interp(z, z_grid, I_grid)
    DM_IGM_z    = prefactor * Ob * H0**2 * I_z
    DM_IGM_corr = DM_IGM_z * (km_to_pc / (m2_to_cm2 * Mpc_to_cm))
    return DM_IGM_corr


# ====================================================
# Quick test
# ====================================================

print(f"DM_IGM_ST(z_frb, n={n}) =",
       DM_IGM_ST(z_frb, 70, 0.3, 0.0495, 5.0))

DM_IGM_ST(z_frb, n=1) = [   7.23667992   16.20912738   20.22618806   25.48472966   25.86442359
   36.65600476   36.92609468   40.27833608   51.89946929   55.18494167
   55.53446001   56.84515377   60.44378329   61.60401224   61.76222528
   62.99277114   67.7391623    77.54123633   80.28149645   81.52026101
   85.1642537    91.38570464   93.83322938   93.96724465   95.84345841
   99.32785539  102.79438367  103.34831344  103.43765696  106.84123807
  106.96694779  110.61252947  111.06149273  113.12672373  113.75527229
  121.31785756  124.11484921  128.53590054  138.05019818  138.1408406
  139.26390021  141.58631518  156.53013656  162.65589517  169.89550885
  170.62990642  170.74924602  181.47553578  186.63622852  188.01855693
  190.78475854  204.2018371   208.84296759  211.16353284  213.94821113
  215.71091278  216.55192203  217.39013724  219.06656764  224.18899387
  224.37526391  235.39003864  243.3417079   248.40251811  250.37061097
  251.49523546  256.16226468  258.3531844   262.300787

In [6]:
# # ====================================================
# # Worker function (module level — required for pickle)
# # ====================================================

# def _grid_worker_ST(args):
#     """
#     Computes H(z) for a single (H0, Om0, lbd) grid point.
#     The grid is always built in LINEAR lbd space.
#     Returns (i, j, k, H_values) or (i, j, k, None) on failure.
#     """
    
#     i, j, k, H0, Om0, lbd, z_pts = args
#     Hvals = H_ST_z(z_pts, H0, Om0, lbd)
#     if Hvals is not None and np.all(np.isfinite(Hvals)):
#         return (i, j, k, Hvals)
#     else:
#         return (i, j, k, None)


# # ====================================================
# # Parallel grid builder
# # ====================================================

# def build_H_grid_parallel_ST(H0_pts, Om0_pts, lbd_pts, z_pts,
#                               n_cores=None, verbose=True):
#     nH    = len(H0_pts)
#     nOm   = len(Om0_pts)
#     nlbd  = len(lbd_pts)
#     nz    = len(z_pts)
#     total = nH * nOm * nlbd
#     n_cores = n_cores or os.cpu_count()
#     tasks = [
#         (i, j, k, H0, Om0, lbd, z_pts)
#         for i, H0  in enumerate(H0_pts)
#         for j, Om0 in enumerate(Om0_pts)
#         for k, lbd in enumerate(lbd_pts)
#     ]
#     H_array   = np.full((nH, nOm, nlbd, nz), np.nan)
#     nan_count = 0
#     if verbose:
#         print(f"  ST model index    : n = {n}")
#         print(f"  Total grid points : {total} ({nH}×{nOm}×{nlbd})")
#         print(f"  Workers           : {n_cores}")
#         print(f"  {'PCT':>5}  {'Done':>8}  {'Elapsed':>9}  {'ETA':>9}  {'Rate':>12}")
#         print("  " + "-"*55)
#     t0       = time.time()
#     count    = 0
#     last_pct = -1
#     chunksize = max(1, total // (n_cores * 20))
#     with Pool(processes=n_cores) as pool:
#         for result in pool.imap_unordered(_grid_worker_ST, tasks,
#                                           chunksize=chunksize):
#             i, j, k, Hvals = result
#             if Hvals is not None:
#                 H_array[i, j, k, :] = Hvals
#             else:
#                 nan_count += 1
#             count   += 1
#             elapsed  = time.time() - t0
#             pct      = int(count / total * 100)
#             if verbose and (pct >= last_pct + 10 or count == total):
#                 eta  = (elapsed / count) * (total - count) if count < total else 0.0
#                 rate = count / elapsed if elapsed > 0 else 0.0
#                 print(f"  {pct:>4}%  {count:>8}/{total}  "
#                       f"{elapsed:>7.1f}s  "
#                       f"ETA {eta:>6.1f}s  "
#                       f"{rate:>8.1f} pts/s")
#                 last_pct = pct
#     nan_frac = nan_count / total
#     if nan_frac > 0:
#         print(f"\n  ⚠  {100*nan_frac:.1f}% NaN in grid — "
#               f"check prior bounds or ODE stability.")
#     else:
#         print(f"\n  ✓  Grid complete with no NaN points.")
#     total_time = time.time() - t0
#     print(f"  Total time: {total_time:.1f}s ({total_time/60:.2f} min)")
#     return H_array


# # ====================================================
# # Interpolation error validation
# # ====================================================

# def validate_interpolation_error_ST(H_interp_4D, H0_pts, Om0_pts, lbd_pts, z_pts,
#                                     n_test=300, tol_percent=0.5):
#     """
#     Validates the interpolator against direct ODE solutions at
#     n_test random interior points (sampled in LINEAR lbd space).
#     """
    
#     rng_v  = np.random.default_rng(0)
#     errors = []
#     for _ in range(n_test):
#         H0_t    = rng_v.uniform(H0_pts[1],   H0_pts[-2])
#         Om0_t   = rng_v.uniform(Om0_pts[1],  Om0_pts[-2])
#         lbd_t   = rng_v.uniform(lbd_pts[1],  lbd_pts[-2])
#         H_true  = H_ST_z(z_pts, H0_t, Om0_t, lbd_t)
#         if H_true is None or not np.all(np.isfinite(H_true)):
#             continue
#         pts      = np.column_stack([np.full(len(z_pts), H0_t),
#                                     np.full(len(z_pts), Om0_t),
#                                     np.full(len(z_pts), lbd_t),
#                                     z_pts])
#         H_interp = H_interp_4D(pts)
#         rel_err  = np.abs((H_true - H_interp) / H_true) * 100.0
#         errors.append(np.max(rel_err))
#     errors = np.array(errors)
#     print(f"\n  Interpolation validation ({n_test} random interior points):")
#     print(f"  Max error    : {errors.max():.4f}%")
#     print(f"  Median error : {np.median(errors):.4f}%")
#     print(f"  95th pct     : {np.percentile(errors, 95):.4f}%")
#     if errors.max() < tol_percent:
#         print(f"  ✓  Interpolator approved (max error < {tol_percent}%). "
#               f"Ready for MCMC.")
#     else:
#         print(f"  ✗  Max error exceeds {tol_percent}%! "
#               f"Increase grid resolution (n_H0, n_Om0, n_lbd).")
#     return errors


# # ====================================================
# # 4D interpolation grid parameters
# # ====================================================

# n_H0  = 30
# n_Om0 = 30
# n_lbd = 30

# H0_grid_pts  = np.linspace(H0_min,  H0_max,  n_H0)
# Om0_grid_pts = np.linspace(Om0_min, Om0_max, n_Om0)
# lbd_grid_pts = np.linspace(lbd_min, lbd_max, n_lbd)


# # ====================================================
# # Build grid (parallel)
# # ====================================================

# print(f"\n=== Building 4D interpolation grid (parallel) | ST n={n} ===")
# H_array = build_H_grid_parallel_ST(
#     H0_grid_pts, Om0_grid_pts, lbd_grid_pts, z_grid,
#     n_cores=os.cpu_count()
# )


# # ====================================================
# # Save grid for reuse
# # ====================================================

# np.save(f"H_array_ST_n{n}_{n_lbd}_grid.npy", H_array)


# # ====================================================
# # Build interpolator
# # ====================================================

# H_interp_4D = RegularGridInterpolator(
#     (H0_grid_pts, Om0_grid_pts, lbd_grid_pts, z_grid),
#     H_array,
#     method="linear",
#     bounds_error=False,
#     fill_value=np.nan
# )


# # ====================================================
# # Validate interpolator
# # ====================================================

# print(f"\n=== Validating interpolator | ST n={n} ===")
# errors = validate_interpolation_error_ST(
#     H_interp_4D, H0_grid_pts, Om0_grid_pts, lbd_grid_pts, z_grid
# )

In [7]:
# =====================================================
# Load and validation of the grid
# =====================================================

# 4D interpolation grid parameters
n_H0  = 30
n_Om0 = 30
n_lbd = 30

H0_grid_pts  = np.linspace(H0_min,  H0_max,  n_H0)
Om0_grid_pts = np.linspace(Om0_min, Om0_max, n_Om0)
lbd_grid_pts = np.linspace(lbd_min, lbd_max, n_lbd)

# Load grid from file
H_array_path = "/home/brunowesley/projetos/MCMC-cosmo/Codes/MG-based/Starobinsky/CC/H_array_ST_n1_30_grid.npy"
H_array = np.load(H_array_path)

# Build interpolator
H_interp_4D = RegularGridInterpolator(
    (H0_grid_pts, Om0_grid_pts, lbd_grid_pts, z_grid),
    H_array,
    method="linear",
    bounds_error=False,
    fill_value=np.nan
)


# ====================================================
# H(z) via interpolator
# ====================================================

def H_ST_interp(z_eval, H0, Om0, lbd):
    """Query interpolator with physical (linear) lbd."""
    
    pts = np.column_stack([
        np.full(len(z_eval), H0),
        np.full(len(z_eval), Om0),
        np.full(len(z_eval), lbd),
        z_eval
    ])
    return H_interp_4D(pts)


# ====================================================
# Interpolated DM_IGM
# ====================================================

def DM_IGM_ST_interp(z, H0, Om0, Ob, lbd):
    prefactor   = 3.0 * c_si * f_IGM * chi_e / (8.0 * np.pi * G_si * mp_si)
    H_grid      = H_ST_interp(z_grid, H0, Om0, lbd)
    integrand   = (1 + z_grid) / H_grid
    I_grid      = cumulative_trapezoid(integrand, z_grid, initial=0.0)
    I_z         = np.interp(z, z_grid, I_grid)
    DM_IGM_z    = prefactor * Ob * H0**2 * I_z
    DM_IGM_corr = DM_IGM_z * (km_to_pc / (m2_to_cm2 * Mpc_to_cm))
    return DM_IGM_corr


# ====================================================
# Quick test
# ====================================================

print(f"DM_IGM_ST_interp(z_frb, n={n}) =",
        DM_IGM_ST_interp(z_frb, 70, 0.3, 0.0495, 5.0))

DM_IGM_ST_interp(z_frb, n=1) = [   7.23669286   16.20915442   20.22622066   25.48476856   25.86446288
   36.65605433   36.92614443   40.27838817   51.89952793   55.18500172
   55.53452021   56.84521455   60.44384503   61.60407424   61.76228731
   62.99283344   67.73922565   77.54130061   80.28156091   81.52032554
   85.16431817   91.385769     93.83329368   93.96730894   95.84352263
   99.32791948  102.79444763  103.34837739  103.4377209   106.84130198
  106.96701169  110.61259342  111.06155669  113.12678771  113.75533629
  121.31792216  124.11491419  128.53596611  138.05026676  138.14090921
  139.26396918  141.58638498  156.53021627  162.65598059  169.89560271
  170.63000126  170.74934103  181.4756468   186.63634891  188.01867984
  190.7848867   204.20199776  208.84314199  211.1637141   213.94840064
  215.71110787  216.55212016  217.3903384   219.06677487  224.18921964
  224.37549036  235.39031051  243.34201668  248.40285268  250.37095557
  251.49558579  256.16264161  258.35357426  26

In [ ]:
# ====================================================
# FRB likelihood
# ====================================================

# Log-prior
def lnprior_frb(theta_frb):
    H0, Om0, Ob, lbd, sigma_host, exp_mu = theta_frb
    if not (H0_min       <= H0         <= H0_max):       return -np.inf
    if not (Om0_min      <= Om0        <= Om0_max):      return -np.inf
    if not (Ob_min       <= Ob         <= Ob_max):       return -np.inf
    if not (lbd_min      <= lbd        <= lbd_max):      return -np.inf
    if not (sig_host_min <= sigma_host <= sig_host_max): return -np.inf
    if not (e_mu_min     <= exp_mu     <= e_mu_max):     return -np.inf
    return 0.0


# Convolved FRB PDF: IGM (Gaussian) + host galaxy (log-normal)
def P_DM_ext_vectorized(z_frb, DM_ext_obs, DM_IGM_mean, sigma_IGM,
                         sigma_host, exp_mu, npts=300):
    mu_host      = np.log(exp_mu)
    upper        = max(DM_ext_obs.max(), 5.0 * exp_mu)
    DM_hosp_grid = np.linspace(1e-2, upper, npts)
    DM_hosp_grid = DM_hosp_grid[None, :]                     
    DM_ext_array      = DM_ext_obs[:, None]
    DM_IGM_array      = DM_ext_array - DM_hosp_grid / (1 + z_frb[:, None])
    sigma_IGM_array   = sigma_IGM[:, None]
    DM_IGM_mean_array = DM_IGM_mean[:, None]

    # standardized variable
    y = (DM_IGM_array - DM_IGM_mean_array) / sigma_IGM_array

    # standard normal PDF
    phi = (1/np.sqrt(2*np.pi)) * np.exp(-0.5 * y**2)

    # normalization factor: 1 - Φ(0 | μ, σ) = Φ(μ/σ)
    norm_factor = 0.5 * (1 + erf(DM_IGM_mean_array / (sigma_IGM_array * np.sqrt(2))))

    # truncated Gaussian
    P_IGM = (1 / sigma_IGM_array) * phi / norm_factor
    P_IGM[DM_IGM_array < 0] = 0.0

    DM_hosp_array = DM_hosp_grid
    P_host = (1.0 / (np.sqrt(2*np.pi) * sigma_host * DM_hosp_array)) \
             * np.exp(-0.5 * ((np.log(DM_hosp_array) - mu_host) / sigma_host)**2)

    integrand = P_IGM * P_host               
    P_total   = trapezoid(integrand, x=DM_hosp_grid[0], axis=1)
    return np.maximum(P_total, 1e-100)


# Log-likelihood
def lnlike_frb(theta_frb, z_frb, DM_ext_obs, npts=300):
    H0, Om0, Ob, lbd, sigma_host, exp_mu = theta_frb
    DM_IGM_mean = DM_IGM_ST_interp(z_frb, H0, Om0, Ob, lbd)
    if not np.all(np.isfinite(DM_IGM_mean)):
        return -np.inf
    sigma_IGM = 173.8 * (z_frb**0.4)
    P_total = P_DM_ext_vectorized(z_frb, DM_ext_obs, DM_IGM_mean,
                                   sigma_IGM, sigma_host, exp_mu,
                                   npts=npts)
    return np.sum(np.log(P_total))


# Log-posterior
def lnprob_frb(theta_frb, z_frb, DM_ext_obs):
    lp = lnprior_frb(theta_frb)
    if not np.isfinite(lp):
        return -np.inf
    return lp + lnlike_frb(theta_frb, z_frb, DM_ext_obs, npts=300)


# ====================================================
# Quick test
# ====================================================

theta_frb_test = [70.0, 0.3, 0.0495, 5.0, 0.7, 120.0]
print("FRB log-posterior =",
       lnprob_frb(theta_frb_test, z_frb, DM_ext_obs))

FRB log-posterior = -650.1407013543017


In [9]:
# ====================================================
# Observable error analysis (FRB)
# ====================================================

def analyze_DM_error_frb(H0, Om0, Ob, lbd):
    DM_true   = DM_IGM_ST(z_frb, H0, Om0, Ob, lbd)
    DM_interp = DM_IGM_ST_interp(z_frb, H0, Om0, Ob, lbd)

    if DM_true is None or DM_interp is None:
        print("Model evaluation failed.")
        return None

    # --- Errors ---
    abs_error = np.abs(DM_interp - DM_true)
    rel_error = abs_error / np.abs(DM_true)
    pct_error = 100.0 * rel_error

    # --- Likelihood impact ---
    theta = [H0, Om0, Ob, lbd, 0.7, 120.0]

    lnL_true = lnlike_frb(theta, z_frb, DM_ext_obs, npts=300)

    # Força avaliação com o modelo "true"
    def lnlike_true(theta_frb):
        H0_, Om0_, Ob_, lbd_, sigma_host_, exp_mu_ = theta_frb
        DM_IGM_mean = DM_IGM_ST(z_frb, H0_, Om0_, Ob_, lbd_)
        if not np.all(np.isfinite(DM_IGM_mean)):
            return -np.inf
        sigma_IGM = 173.8 * (z_frb**0.4)
        P_total = P_DM_ext_vectorized(
            z_frb, DM_ext_obs, DM_IGM_mean,
            sigma_IGM, sigma_host_, exp_mu_,
            npts=300
        )
        return np.sum(np.log(P_total))

    lnL_true   = lnlike_true(theta)
    lnL_interp = lnlike_frb(theta, z_frb, DM_ext_obs, npts=300)

    delta_chi2 = -2.0 * (lnL_interp - lnL_true)

    # --- Output ---
    results = {
        "Metric": [
            "Max absolute error [pc/cm^3]",
            "Median absolute error [pc/cm^3]",
            "Max relative error",
            "Median relative error",
            "Max percent error [%]",
            "Median percent error [%]"
        ],
        "Value": [
            np.max(abs_error),
            np.median(abs_error),
            np.max(rel_error),
            np.median(rel_error),
            np.max(pct_error),
            np.median(pct_error)
        ]
    }

    df_results = pd.DataFrame(results)

    print(f"\n=== Observable Error Summary (FRB | ST | DM_IGM) ===\n")
    print(df_results.to_string(index=False))

    print("\n=== Likelihood Impact ===\n")
    print(f"lnL_true        : {lnL_true:.6e}")
    print(f"lnL_interp      : {lnL_interp:.6e}")
    print(f"Delta chi^2     : {delta_chi2:.6e}")

    return df_results

_ = analyze_DM_error_frb(H0=70.0, Om0=0.3, Ob=0.0495, lbd=5.0)


=== Observable Error Summary (FRB | ST | DM_IGM) ===

                         Metric    Value
   Max absolute error [pc/cm^3] 0.035345
Median absolute error [pc/cm^3] 0.000168
             Max relative error 0.000028
          Median relative error 0.000001
          Max percent error [%] 0.002759
       Median percent error [%] 0.000111

=== Likelihood Impact ===

lnL_true        : -6.501410e+02
lnL_interp      : -6.501407e+02
Delta chi^2     : -5.252652e-04


In [ ]:
# ====================================================
# MCMC configuration
# ====================================================

ndim     = 6
nwalkers = 48
nsteps   = 30000
n_cores = os.cpu_count()

theta0 = np.array([70.0, 0.3, 0.0495, 5.0, 0.7, 120.0])
scales = np.array([ 2.0, 0.02, 0.001, 1.0, 0.1,  10.0])

rng = np.random.default_rng(42)
p0  = theta0 + scales * rng.standard_normal((nwalkers, ndim))
for i in range(nwalkers):
    while not np.isfinite(lnprior_frb(p0[i])):
        p0[i] = theta0 + scales * rng.standard_normal(ndim)


# ====================================================
# MCMC run
# ====================================================

print(f"\n=== Running MCMC (ST n={n} | FRB | Uniform on lbd) ===")
t_mcmc = time.time()

with Pool(processes=n_cores) as pool:
    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, lnprob_frb,
        args=(z_frb, DM_ext_obs),
        pool=pool
    )
    sampler.run_mcmc(p0, nsteps, progress=True)
    
print(f"MCMC completed in {(time.time() - t_mcmc)/60:.2f} min")

In [ ]:
# ====================================================
# Convergence diagnostics
# ====================================================

print("\n=== Convergence diagnostics ===")
chain        = sampler.get_chain()
nsteps_total = chain.shape[0]
acc_frac     = np.mean(sampler.acceptance_fraction)
print(f"Mean acceptance fraction: {acc_frac:.3f}")
if   0.2  <= acc_frac <= 0.5:  acc_status = "IDEAL"
elif 0.15 <= acc_frac <= 0.6:  acc_status = "ACCEPTABLE"
else:                           acc_status = "PROBLEMATIC"
print(f"Acceptance status: {acc_status}")

tau          = sampler.get_autocorr_time(tol=0)
tau_max      = np.max(tau)
length_ratio = nsteps_total / tau_max
print(f"\nMax autocorrelation time τ_max = {tau_max:.2f}")
print(f"Chain length / τ_max           = {length_ratio:.1f}")
length_status = "PASSED" if length_ratio >= 50 else "FAILED"
print(f"Length criterion (>50×τ)       : {length_status}")

nburn  = int(3 * tau_max)
nthin  = max(1, int(tau_max / 2))
n_post = nsteps_total - nburn
N_eff  = nwalkers * (n_post / tau)
N_eff_min = np.min(N_eff)
print(f"\nBurn-in (3×τ_max)              = {nburn} steps ({100*nburn/nsteps_total:.1f}%)")
print(f"Thin (τ_max/2)                 = {nthin}")
print(f"Minimum N_eff                  = {int(N_eff_min)}")
if   N_eff_min >= 2000: eff_status = "EXCELLENT"
elif N_eff_min >= 1000: eff_status = "VERY GOOD"
elif N_eff_min >= 500:  eff_status = "ACCEPTABLE"
else:                   eff_status = "LOW"
print(f"Sampling quality               : {eff_status}")

if acc_status != "PROBLEMATIC" and length_status == "PASSED" and N_eff_min >= 500:
    print("\nCONVERGENCE STATUS: ✅ PASSED")
else:
    print("\nCONVERGENCE STATUS: ❌ NOT RELIABLE")

In [ ]:
# ====================================================
# Chain extraction
# ====================================================

# flat_samples = sampler.get_chain(discard=nburn, thin=nthin, flat=True)    # Apply thin
flat_samples = sampler.get_chain(discard=nburn, flat=True)                  # No thin

# Save
fname       = f"flat_samples_ST_n{n}_frb.npy"
np.save(fname,       flat_samples)

print(f"\nFinal posterior samples : {len(flat_samples)}")
print(f"Burn-in used            : {nburn} steps")
print(f"Saved to                : {fname}")

In [ ]:
# ====================================================
# Corner plot (GetDist) 
# ====================================================

param_names  = ["H0", "Om0", "Ob", "lbd", "sigma_host", "exp_mu"]
param_labels = [r"H_0", r"\Omega_{\rm m}", r"\Omega_{\rm b}", r"\lambda_{\rm S}",
                r"\sigma_{\rm host}", r"e^\mu"]
fmt_map      = {"H0": ".2f", "Om0": ".3f", "Ob": ".4f", "lbd": ".2f", 
                "sigma_host": ".2f", "exp_mu": ".1f"}

samples_gd = MCSamples(
    samples=flat_samples,          
    names=param_names,
    labels=param_labels
)

samples_gd.updateSettings({
    "smooth_scale_1D": 0.25,
    "smooth_scale_2D": 0.25,
    "fine_bins":       1024,
    "fine_bins_2D":    1024,
})

g = plots.get_subplot_plotter()
g.settings.axes_fontsize       = 14
g.settings.lab_fontsize        = 16
g.settings.legend_fontsize     = 12
g.settings.linewidth_contour   = 1.5
g.settings.num_plot_contours   = 2
g.settings.axis_marker_lw      = 1.0
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add    = 0.3

g.triangle_plot(samples_gd, filled=True,
                legend_labels=[f"FRB (ST n={n})"],
                title_limit=0)

# Add titles with median ± 68% CL
for i, name in enumerate(param_names):
    samp   = flat_samples[:, i]
    median  = np.percentile(samp, 50)
    lower68 = np.percentile(samp, 16)
    upper68 = np.percentile(samp, 84)
    ep68    = upper68 - median
    em68    = median  - lower68
    ax      = g.subplots[i, i]
    line    = ax.get_lines()[0]
    line.set_linewidth(1.0)
    ax.axvline(median, color="blue", lw=1.0, ls="--", alpha=0.9)
    fmt   = fmt_map[name]
    title = f"${param_labels[i]} = {median:{fmt}}^{{+{ep68:{fmt}}}}_{{-{em68:{fmt}}}}$"
    ax.set_title(title, fontsize=11, pad=4)

plt.subplots_adjust(top=0.95)
figname = f"Corner_ST_n{n}_frb.png"
plt.savefig(figname, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ====================================================
# Marginalized statistics
# ====================================================

print("\n" + "="*60)
print(f"{'Final Inference Results (ST n='+str(n)+' | FRB data)':^60}")
print("="*60)

for name, idx, fmt in [("H0", 0, ".2f"), ("Om0", 1, ".3f"), ("Ob", 2, ".4f"),
                       ("lbd", 3, ".2f"), ("sigma_host", 4, ".2f"), ("exp_mu", 5, ".1f")]:
    samp    = flat_samples[:, idx]
    median  = np.percentile(samp, 50)
    lower68 = np.percentile(samp, 16)
    upper68 = np.percentile(samp, 84)
    lower95 = np.percentile(samp, 2.5)
    upper95 = np.percentile(samp, 97.5)
    ep68 = upper68 - median;  em68 = median - lower68
    ep95 = upper95 - median;  em95 = median - lower95
    print(f"\n  {name}:")
    print(f"    {median:{fmt}} +{ep68:{fmt}} -{em68:{fmt}}  (68% CL)")
    print(f"    {median:{fmt}} +{ep95:{fmt}} -{em95:.{fmt[1:]}}")

print("\n" + "="*60)

In [ ]:
# ====================================================
# Walker chains
# ====================================================

chain_plot   = sampler.get_chain().copy()
chain_labels = [r"$H_0$", r"$\Omega_{\rm m}$", r"$\Omega_{\rm b}$",
                r"$\lambda_{\rm S}$", r"$\sigma_{\rm host}$", r"$e^\mu$"]

fig, axes = plt.subplots(6, figsize=(10, 7), sharex=True)

for i, label in enumerate(chain_labels):
    axes[i].plot(chain_plot[:, :, i], alpha=0.3, lw=0.5)
    axes[i].set_ylabel(label)
axes[-1].set_xlabel("Step")

plt.suptitle(f"Walker chains — ST n={n} | FRB | Uniform prior on lbd",
             fontsize=11)

plt.tight_layout()
plt.show()